# H2 · Six-qubit QAOA experiment with an exact classical reference

Use **Lunar H0 (Python 3.12)** with the pinned notebook requirements. Everything here runs
locally: **no provider API, hardware job, paid execution, noise model or repair**.
The synthetic three-mission problem is deliberately small enough to solve classically.

QAOA follows alternating cost phases and X mixing from
[Farhi, Goldstone and Gutmann (2014)](https://arxiv.org/abs/1411.4028).
See also [IBM's QAOA ansatz definition](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.library.QAOAAnsatz).
The simulator uses NumPy statevectors; this notebook does not invoke Qiskit or IBM hardware.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
from html import escape
import os
import sys
import numpy as np
from IPython.display import HTML, display

start = Path(os.environ.get("LUNAR_REPO_ROOT", Path.cwd())).resolve()
roots = [start, *start.parents, start / "quantum-hybrid-workspace", start / "h0-hybrid-workspace"]
ROOT = next((p for p in roots if (p / "backend/app/hybrid_quantum.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open from the repository or set LUNAR_REPO_ROOT before starting the kernel.")
sys.path.insert(0, str(ROOT))
from shared.hybrid import Scenario, load_scenario
from backend.app.hybrid_planning import solve_exact, validate_plan
from backend.app.hybrid_quantum import (
    QaoaResult, compile_qubo, audit_mapping, compact_model, run_qaoa,
)
import backend.app.hybrid_quantum as quantum_module
assert Path(quantum_module.__file__).resolve() == ROOT / "backend/app/hybrid_quantum.py"
print("Python:", sys.version.split()[0], "NumPy:", np.__version__)
print("Mode: local noiseless statevector simulation; hardware jobs: 0")

def table(headers, rows):
    def row(values, tag):
        return "<tr>" + "".join(f"<{tag}>{escape(str(v))}</{tag}>" for v in values) + "</tr>"
    display(HTML("<table>" + row(headers, "th") + "".join(row(r, "td") for r in rows) + "</table>"))

Python: 3.12.4 NumPy: 2.5.4
Mode: local noiseless statevector simulation; hardware jobs: 0


## 1. Preserve the classical answer

The classical exact solver is our reference. Quantum tuning does not replace it or make
this tiny problem faster. Optimal probability refers to sampling *any* of the six equally
optimal feasible plans, not just the deterministic classical representative.

In [2]:
scenario = load_scenario(ROOT / "shared/examples/hybrid/shared-conflict.json")
classical = solve_exact(scenario)
assert classical.validation.exact_score == "13/5"
print("Classical optimum:", classical.validation.score)
print("Feasible plans:", classical.feasible_count, "/", classical.total_combinations)
print("Equal optima:", classical.optimal_plan_count)
print("Classical elapsed seconds:", classical.elapsed_s)

Classical optimum: 2.6
Feasible plans: 24 / 64
Equal optima: 6
Classical elapsed seconds: 0.00463645799754886


## 2. Build and audit the 12-variable QUBO

One binary variable per mission option gives 12 variables. The energy is negative score
plus exactly-one, forbidden-option and pair-conflict penalties. The penalty is strictly
larger than the total absolute candidate-score bound. Coefficients are exact rational strings.

The compiler checks every one-choice assignment against H1. The audit examines all
**4096 binary assignments**, including invalid selections, and compares ground states
with the original constrained optimum. Unsupported higher-order capacity rules fail explicitly.

In [3]:
qubo = compile_qubo(scenario)
audit = audit_mapping(qubo, scenario)
assert audit.ground_matches_classical
assert audit.binary_assignments == 4096 and audit.optimal_assignments == 6
print("Binary variables:", len(qubo.option_ids))
print("Penalty:", qubo.penalty, "Offset:", qubo.offset)
print("Audit:", audit.model_dump())

Binary variables: 12
Penalty: 53/5 Offset: 159/5
Audit: {'binary_assignments': 4096, 'one_choice_assignments': 64, 'feasible_assignments': 24, 'optimal_assignments': 6, 'ground_energy': '-13/5', 'ground_matches_classical': True}


## 3. Encode the choice domain with six qubits

Four choices per mission need two binary register bits; three missions need six qubits.
All 64 register states represent exactly one choice per mission. The cost diagonal is the
12-variable QUBO restricted to that domain, centered and scaled for tuning.

**This is not a six-variable quadratic QUBO.** Compact encoding introduces up to four-body
Pauli-Z terms. Fewer qubits can require more complex gates. The counts below assume logical
all-to-all connectivity and a simple parity-chain decomposition; hardware routing, native
gate synthesis, noise and transpiled depth are not included.

In [4]:
compact = compact_model(qubo)
assert compact.logical_qubits == 6
print("QUBO variables:", len(qubo.option_ids), "simulated qubits:", compact.logical_qubits)
print("Basis states:", len(compact.one_hot_masks))
print("Largest Z-term locality:", compact.max_z_locality)
print("Logical CX estimate per cost layer:", compact.cx_per_cost_layer)
print("RZ estimate per cost layer:", sum(mask != 0 for mask, _ in compact.z_terms))
print("Energy shift:", compact.energy_shift, "energy scale:", compact.energy_scale)

QUBO variables: 12 simulated qubits: 6
Basis states: 64
Largest Z-term locality: 4
Logical CX estimate per cost layer: 30
RZ estimate per cost layer: 15
Energy shift: 5.55 energy scale: 24.45


## 4. Run the low-depth default

Start from a uniform state, alternate cost phases with X mixers, and minimize expected
normalized energy with a bounded seeded classical search. Half the budget explores random
angles; the remainder refines the incumbent. The uniform state is the first evaluated point.

The optimizer uses exact simulator expectations, **not 512 shots per tuning evaluation**.
The 512 shots below are only final multinomial samples of the optimized distribution.
Transferring this loop to hardware will require a separate measurement/time/cost budget.

In [5]:
result = run_qaoa(scenario, depth=1, evaluations=160, shots=512, seed=17)
assert result.logical_qubits == 6 and result.hardware_jobs == 0
print("Gammas (radians):", result.gammas)
print("Betas (radians):", result.betas)
print("Expected normalized energy:", result.expected_normalized_energy)
print("Objective evaluations:", result.objective_evaluations)
print("Elapsed seconds (including mapping/audit):", result.elapsed_s)
table(["Distribution", "Feasible probability", "Optimal probability"], [
    ["Uniform", result.uniform_feasible_probability, result.uniform_optimal_probability],
    ["QAOA exact simulator distribution", result.feasible_probability, result.optimal_probability],
    ["Observed in 512 final samples", result.sampled_feasible_rate, result.sampled_optimal_rate],
])

Gammas (radians): (2.2129981066869426,)
Betas (radians): (-0.228844809080558,)
Expected normalized energy: -0.2105608930516292
Objective evaluations: 160
Elapsed seconds (including mapping/audit): 13.867858292003802


Distribution,Feasible probability,Optimal probability
Uniform,0.375,0.09375
QAOA exact simulator distribution,0.8059442060272193,0.2052098666166589
Observed in 512 final samples,0.826171875,0.1953125


## 5. Inspect raw outcomes

Each outcome is checked against H1's original constraints. Invalid samples remain invalid;
there is no hidden repair or post-selection. Probabilities and sampled counts are separate.
Zero-count outcomes are retained in the full result. The table shows the 12 most probable.

In [6]:
table(["Basis index", "Probability", "Count", "Feasible", "Optimal", "Score", "Selected options"], [
    [s.basis_index, round(s.probability, 6), s.count, s.feasible, s.optimal, s.score, ", ".join(s.selected_option_ids)]
    for s in sorted(result.samples, key=lambda sample: -sample.probability)[:12]
])
assert sum(s.count for s in result.samples) == 512
assert len(result.samples) == 64
assert not result.repair_applied
if result.best_sampled_plan:
    report = validate_plan(result.best_sampled_plan, scenario)
    assert report.status == "feasible"
    print("Best sampled feasible score:", report.score)
else:
    print("No feasible plan sampled; this is reported explicitly.")

Basis index,Probability,Count,Feasible,Optimal,Score,Selected options
9,0.034202,18,True,True,2.6,"mission-1-option-2, mission-2-option-3, mission-3-option-1"
6,0.034202,15,True,True,2.6,"mission-1-option-3, mission-2-option-2, mission-3-option-1"
18,0.034202,20,True,True,2.6,"mission-1-option-3, mission-2-option-1, mission-3-option-2"
24,0.034202,21,True,True,2.6,"mission-1-option-1, mission-2-option-3, mission-3-option-2"
33,0.034202,17,True,True,2.6,"mission-1-option-2, mission-2-option-1, mission-3-option-3"
36,0.034202,9,True,True,2.6,"mission-1-option-1, mission-2-option-2, mission-3-option-3"
13,0.033886,31,True,False,2.5,"mission-1-option-2, mission-2-option-4, mission-3-option-1"
7,0.033886,16,True,False,2.5,"mission-1-option-4, mission-2-option-2, mission-3-option-1"
19,0.033886,19,True,False,2.5,"mission-1-option-4, mission-2-option-1, mission-3-option-2"
28,0.033886,15,True,False,2.5,"mission-1-option-1, mission-2-option-4, mission-3-option-2"


Best sampled feasible score: 2.6


## 6. Compare one and two layers across three fixed seeds

Fixed settings: 160 simulator objective evaluations, 512 final shots, seeds 17/23/42.
The acceptance criterion was declared before the initial run: mean feasible and optimal
probability exceed uniform, and each run samples a feasible plan. This is a small synthetic
sanity check, not a held-out generalization study or evidence of quantum advantage.

In [7]:
runs = [result]
for depth in (1, 2):
    for seed in (17, 23, 42):
        if depth == 1 and seed == 17:
            continue
        runs.append(run_qaoa(scenario, depth=depth, evaluations=160, shots=512, seed=seed))
table(["Layers", "Seed", "Feasible probability", "Optimal probability", "Observed optimal rate", "Logical CX", "Elapsed s"], [
    [r.depth, r.seed, round(r.feasible_probability, 6), round(r.optimal_probability, 6),
     round(r.sampled_optimal_rate, 6), r.cx_gate_estimate, round(r.elapsed_s, 4)]
    for r in sorted(runs, key=lambda r: (r.depth, r.seed))
])
for depth in (1, 2):
    group = [r for r in runs if r.depth == depth]
    mean_feasible = sum(r.feasible_probability for r in group) / len(group)
    mean_optimal = sum(r.optimal_probability for r in group) / len(group)
    passed = mean_feasible > 24/64 and mean_optimal > 6/64 and all(r.best_sampled_plan for r in group)
    print(f"Depth {depth}: mean feasible={mean_feasible:.6f}, mean optimal={mean_optimal:.6f}, acceptance={bool(passed)}")

Layers,Seed,Feasible probability,Optimal probability,Observed optimal rate,Logical CX,Elapsed s
1,17,0.805944,0.20521,0.195312,30,13.8679
1,23,0.795934,0.202351,0.21875,30,0.9859
1,42,0.811094,0.198733,0.195312,30,0.9426
2,17,0.955768,0.250328,0.236328,60,0.9469
2,23,0.897702,0.23419,0.25,60,0.945
2,42,0.821416,0.20928,0.205078,60,0.9557


Depth 1: mean feasible=0.804324, mean optimal=0.202098, acceptance=True
Depth 2: mean feasible=0.891629, mean optimal=0.231266, acceptance=True


## 7. Interpret cost honestly

One layer uses half the logical CX/RZ/RX gates of two layers in this construction. It is the
low-depth default, not a proven global minimum-cost circuit. The classical exact solver
already finds the optimum cheaply, while this experiment enumerates/audits the complete
small problem and optimizes full statevectors. There is no speed or monetary advantage claim.

Uniform random sampling with 512 trials also has a very high chance of seeing an optimum.
Improved *per-sample* probability is not automatically a useful *end-to-end* advantage.

In [8]:
table(["Layers", "Qubits", "Estimated CX", "Estimated RZ", "RX", "Max Z locality", "Hardware jobs"], [
    [r.depth, r.logical_qubits, r.cx_gate_estimate, r.rz_gate_estimate, r.rx_gate_count, r.max_z_locality, r.hardware_jobs]
    for r in (next(r for r in runs if r.depth == 1), next(r for r in runs if r.depth == 2))
])
print("Uniform single-shot optimal probability:", 6/64)
print("Uniform probability of missing every optimum in 512 independent trials:", (1-6/64)**512)

Layers,Qubits,Estimated CX,Estimated RZ,RX,Max Z locality,Hardware jobs
1,6,30,15,6,4,0
2,6,60,30,12,4,0


Uniform single-shot optimal probability: 0.09375
Uniform probability of missing every optimum in 512 independent trials: 1.2911777230542295e-22


## 8. Unsupported cases fail explicitly

The bounded compiler supports only instances whose original hard constraints are exactly
unary/pairwise on the choice domain. Capacity two with three overlapping unit demands is
not pairwise: every pair fits but the triple does not. It requires a different encoding
(for example slack variables), rather than silently accepting an invalid plan.

In [9]:
raw = scenario.model_dump(mode="json")
raw["revision"] += 1
raw["resources"][0]["capacity"] = 2
try:
    compile_qubo(Scenario.model_validate(raw))
except ValueError as error:
    print("Expected encoding limit:", error)
else:
    raise AssertionError("Higher-order capacity was silently approximated")
for name in ("missing-capacity", "resource-outage"):
    source = load_scenario(ROOT / "shared/examples/hybrid" / f"{name}.json")
    try:
        compile_qubo(source)
    except ValueError as error:
        print(name, "->", str(error)[:160])
    else:
        raise AssertionError("Unsupported input was accepted")

Expected encoding limit: Resource constraint requires higher-order or slack-variable mapping
missing-capacity -> QUBO input incomplete: (Issue(code='unknown_capacity', message='Required resource capacity is unknown.', option_ids=(), mission_id=None, resource_id='shared-ant
resource-outage -> QUBO input unsupported: (Issue(code='unapplied_events', message='H1 accepts static inputs only; event declarations require explicit state/resource updates befor


## 9. Export full evidence and reproduce

The export contains every basis outcome, raw count, probability, feasibility label,
parameter values, budgets, gate estimates and input identity. Re-check selected plans
against the source scenario before using an imported result. Runtime varies across runs.

In [10]:
with TemporaryDirectory() as folder:
    output = Path(folder) / "h2-qaoa-result.json"
    output.write_text(result.model_dump_json(indent=2), encoding="utf-8")
    restored = QaoaResult.model_validate_json(output.read_text(encoding="utf-8"))
    assert restored == result
    if restored.best_sampled_plan:
        assert validate_plan(restored.best_sampled_plan, scenario).status == "feasible"
print("Full result round trip passed; temporary export removed.")
print("Quantum hardware jobs submitted:", sum(r.hardware_jobs for r in runs))
print("Next: resource integration and broader matched classical/hybrid benchmarks.")

Full result round trip passed; temporary export removed.
Quantum hardware jobs submitted: 0
Next: resource integration and broader matched classical/hybrid benchmarks.
